In [2]:

import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("/Users/hemant/Desktop/dva/project/data/raw/Teen_Mental_Health_Dataset.csv")
df.head(-9)

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1186,18,male,1.6,Both,5.4,2.0,3.20,1.8,low,8,8,3,0
1187,18,male,2.0,Instagram,8.9,0.8,2.96,0.5,medium,6,2,2,0
1188,17,female,7.4,Instagram,7.4,2.0,3.76,1.7,high,10,1,4,0
1189,14,female,1.6,TikTok,4.8,2.4,2.97,1.9,low,9,9,10,0


In [7]:
df.drop(columns=['depression_label'], inplace=True)

In [9]:
df.isnull().sum().sort_values(ascending=False)

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
dtype: int64

In [16]:
# Handle Missing Values Fill numerical with median
num_cols = df.select_dtypes(include=np.number).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [17]:
# Fill categorical with mode
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:

    df[col].fillna(df[col].mode()[0], inplace=True)

/var/folders/26/znjd35k14_3936h0t6_2x8880000gn/T/ipykernel_98587/825739428.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [18]:
# 3. Remove Duplicates
df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

After removing duplicates: (1200, 12)


In [20]:
# Ordinal encoding

interaction_map = {'low': 1, 'medium': 2, 'high': 3}

df['social_interaction_score'] = df['social_interaction_level'].map(interaction_map)

# Mental health composite score

df['mental_health_score'] = df['stress_level'] + df['anxiety_level']

# Risk flags

df['high_social_media_usage'] = (df['daily_social_media_hours'] > 4).astype(int)

df['low_sleep'] = (df['sleep_hours'] < 6).astype(int)

df['late_night_screen'] = (df['screen_time_before_sleep'] > 2).astype(int)

In [21]:
# Cap extreme values (simple approach)

df['daily_social_media_hours'] = df['daily_social_media_hours'].clip(0, 12)

df['sleep_hours'] = df['sleep_hours'].clip(0, 12)

In [22]:
print("\nFinal Shape:", df.shape)

print(df.head())


Final Shape: (1200, 17)
   age  gender  daily_social_media_hours platform_usage  sleep_hours  \
0   14    male                       7.9      Instagram          7.4   
1   19  female                       1.9         TikTok          8.0   
2   17  female                       1.3      Instagram          7.6   
3   15    male                       7.4         TikTok          6.9   
4   15  female                       4.7           Both          4.9   

   screen_time_before_sleep  academic_performance  physical_activity  \
0                       2.9                  3.01                1.5   
1                       2.9                  3.22                0.8   
2                       0.5                  3.92                0.0   
3                       1.6                  3.48                0.8   
4                       3.0                  2.37                1.4   

  social_interaction_level  stress_level  anxiety_level  addiction_level  \
0                      low       

In [24]:
from pathlib import Path

output_path = Path("/Users/hemant/Desktop/dva/project/data/processed/cleaned_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"Saved file to: {output_path}")

Saved file to: /Users/hemant/Desktop/dva/project/data/processed/cleaned_data.csv
